# Scale check: post-convergence steering in a parallel hybrid language model

A standalone Colab notebook for the paper's scale section. It tests whether the phenomenon and its
capacity-limited boundary **replicate** in a larger fused hybrid trained jointly on natural text
and recall. It is a conceptual replication, not the original experiment scaled up: the
architecture is fused, the primary steering signal is availability, and dependence is measured
at route level.

**Model.** A parallel hybrid in the style of Hymba and Falcon-H1. Each block sends one normalised
input to a gated linear recurrence (minGRU/Hawk-style, parallel scan) and to causal RoPE
attention; both writes are added to the residual stream, then an MLP.
- `SIZE="S"` (main): 6 layers, d=256, 6.3M non-embedding / 8.5M total parameters.
- `SIZE="M"`: 8 layers, d=512, 33.6M non-embedding / 37.9M total parameters.

**Data.** TinyStories language modelling mixed with MQAR-style in-context recall at 2–64
key–value pairs. The recall probe is synthetic; natural text enters through the LM loss the
model must preserve.

## Workflow

| Step | Settings | What runs | Send back |
|---|---|---|---|
| 1 | `PRESET="smoke"` | every block with tiny budgets plus the self-test (numbers meaningless) | only errors, if any |
| 2 | `PRESET="full"`, `SEEDS=[0]`, `RUN_INTERVENTIONS=False` | recurrent-only viability, then the free hybrid | `VIABILITY_SUMMARY.txt` |
| 3 | `RUN_INTERVENTIONS=True`, `SEEDS=[0]` | `predictions.json` is written, then the arms for seed 0 | nothing yet |
| 4 | `SEEDS=[0, 1, 2]` | seeds 1 and 2 through every stage; seed 0 is skipped | `RESULTS_SUMMARY.txt` |

Use **Runtime → Run all** each time. Finished work is recorded on Google Drive and skipped. A
disconnect loses at most `CKPT_MINUTES` of training: reconnect, run all again, and it resumes.

## Design

1. **Viability.** A recurrent-only model trained from scratch shows where recurrence alone
   solves each load.
2. **Free hybrid.** Trained to convergence and checkpointed (model, optimiser, scaler) at the
   switch step.
3. **`predictions.json`.** Written once, before any intervention, and never overwritten. A load is
   predicted to hand over if recurrent-only accuracy is at least `VIABLE`; loads the free hybrid
   does not solve are excluded. If the file exists but the configuration has changed, the
   notebook stops rather than mixing results. To start over, delete the file deliberately.
4. **Arms from the switch checkpoint**, all at constant learning rate:
   - `never`
   - `late_availability`: no attention write on half the rows, no penalty term (primary)
   - `late_price`: attention write norm priced, λ set to 10% of the task loss at the switch
   - `delete_finetune`: attention removed, then fine-tuned
5. **`withdrawn`.** Continues `late_availability` with the pressure removed.

**Measurement.**
- **Route-level resample ablation:** every layer's attention (or recurrent) write is replaced by
  the write another sequence produced in its own clean pass. D is normalised by headroom above
  chance, as in the paper.
- **Attention deletion:** recall accuracy and LM loss with no attention at all.
- **Post-switch minimum accuracy:** taken only from checkpoints after the switch, with dense early
  logging.


In [ ]:
import copy, hashlib, itertools, json, math, os, pathlib, random, shutil, subprocess, sys, time
from dataclasses import dataclass, asdict

# ============================== CHOOSE HERE ==============================
PRESET                 = "smoke"   # "smoke": every block, tiny budgets, numbers meaningless | "full"
SIZE                   = "S"       # "S" is the main configuration; "M" only if time remains
SEEDS                  = [0]       # start with [0]; later [0, 1, 2]. Finished work is skipped.
RUN_INTERVENTIONS      = False     # full preset: set True only after checking the load split
ALLOW_DEGENERATE_SPLIT = False     # full preset: permit predictions when loads do not split
USE_DRIVE              = True      # on Colab, keep data, results and checkpoints on Google Drive
DRIVE_DIR              = os.environ.get("RESULTS_DIR", "results/hybrid_scale")
LOCAL_TEXT             = None      # optional TinyStories .txt with <|endoftext|> separators
# =========================================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_DIR
else:
    WORKDIR = os.environ.get("SCALE_WORKDIR", str(pathlib.Path.cwd() / "hybrid_scale"))

for _pkg in ("datasets", "transformers"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
# bf16 needs compute capability >= 8 (L4, A100); T4 and P100 use fp16 with a grad scaler
AMP_DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) \
    if IS_CUDA else torch.bfloat16


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=IS_CUDA)


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


def P(full, smoke):
    return full if PRESET == "full" else smoke


RESUME         = True
PRE            = P(6000, 300)       # free hybrid steps before the switch
POST           = P(3000, 150)       # post-switch steps for every arm
WITHDRAW       = P(1500, 100)       # zero-pressure continuation after late availability
LOG            = P(500, 50)         # evaluation interval for from-scratch runs
LOG_POST       = P(250, 50)         # evaluation interval after the switch
EARLY_POST     = P((25, 50, 100), (10, 25))   # extra early post-switch evaluations (dips)
CKPT_MINUTES   = P(10.0, 2.0)       # mid-run checkpoint interval (wall clock)
N_STORIES      = P(400_000, 20_000)
N_EVAL_BATCHES = P(4, 1)            # x 64 recall sequences per load
N_EVAL_LM      = P(8, 2)            # x 32 validation sequences

LOADS      = [2, 4, 8, 16, 32, 64]  # key-value pairs per recall sequence
P_ABSENT   = 0.5                    # late availability: fraction of rows with no attention write
PRICE_FRAC = 0.1                    # late price: penalty = 10% of task loss at the switch
VIABLE     = 0.90                   # recurrent-only accuracy that predicts handover
SOLVED     = 0.95                   # free-hybrid accuracy needed for a load to enter the claims
RETAIN     = 0.95                   # handover keeps >= 95% of that seed's free accuracy
NEAR       = 0.03                   # recurrent-only within this of VIABLE is flagged as borderline
D_FLOOR    = 0.10                   # D is reported as undefined when accuracy - chance < D_FLOOR


@dataclass
class Cfg:
    d: int = 256
    n_layers: int = 6
    n_heads: int = 4
    rnn_expand: float = 1.0
    mlp_mult: int = 4
    seq_len: int = 256
    batch_lm: int = 32
    batch_rec: int = 16
    rec_groups: int = 4
    recall_w: float = 1.0
    lr: float = 1e-3
    warmup: int = 300
    weight_decay: float = 0.0       # zero, as in the paper, so `never` cannot drift from decay
    grad_clip: float = 1.0
    lm_vocab: int = 8000            # GPT-2 tokens remapped to the 7,999 most frequent + UNK
    n_keys: int = 256
    n_vals: int = 256
    n_query: int = 8


SIZES = {"S": dict(d=256, n_layers=6, n_heads=4, lr=1e-3),
         "M": dict(d=512, n_layers=8, n_heads=8, lr=6e-4)}
CFG = Cfg(**SIZES[SIZE])
if PRESET == "smoke":
    CFG.warmup = 50
assert 2 + 2 * max(LOADS) + 2 * CFG.n_query <= CFG.seq_len
assert CFG.batch_rec % CFG.rec_groups == 0 and (CFG.d // CFG.n_heads) % 2 == 0
assert PRESET in ("smoke", "full") and SIZE in SIZES

UNK = 0
PAD, SEP, QRY = CFG.lm_vocab, CFG.lm_vocab + 1, CFG.lm_vocab + 2
KEY0 = CFG.lm_vocab + 3
VAL0 = KEY0 + CFG.n_keys
VOCAB = VAL0 + CFG.n_vals
IGNORE = -100
CHANCE = 1.0 / CFG.n_vals

TAG = f"{SIZE}_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, CKPTDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}" for n in
                                   ("results", "checkpoints", "figures", "tables"))
DATADIR = WORK / "data"
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR, DATADIR):
    _d.mkdir(parents=True, exist_ok=True)

INTERVENTIONS_ALLOWED = PRESET == "smoke" or RUN_INTERVENTIONS


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


_per_seed = (PRE + POST) + PRE + 4 * POST + WITHDRAW
print(f"colab {IN_COLAB} | device {GPU_NAME} | amp {str(AMP_DTYPE).split('.')[-1]} | torch {torch.__version__}")
print(f"preset {PRESET} | size {SIZE} | seeds {SEEDS} | interventions allowed {INTERVENTIONS_ALLOWED}")
print(f"steps per seed: viability {PRE + POST}, free {PRE}, 4 arms x {POST}, withdrawn {WITHDRAW} "
      f"= {_per_seed:,}")
print(f"work dir {WORK}")
if not IS_CUDA and PRESET == "full":
    print("!! CPU with PRESET='full' will take days. Runtime -> Change runtime type -> GPU.")


In [ ]:
# Persistence. Results: append-only JSONL per block, one record per finished unit, keyed by a
# content hash of its full specification (config included), fsynced on write. Checkpoints:
# written to a temporary file and renamed; the previous checkpoint is kept as .prev so a file
# corrupted by a disconnect mid-write falls back one checkpoint instead of losing the run.

def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    return str(o)


def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


class Store:
    def __init__(self, block):
        self.block, self.path, self.recs = block, RESDIR / f"{block}.jsonl", {}
        if RESUME and self.path.exists():
            bad = 0
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] ignored {bad} truncated record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n"); f.flush(); os.fsync(f.fileno())
        return r

    def frame(self, keys=None):
        out = []
        for k, r in self.recs.items():
            if keys is not None and k not in keys:
                continue
            body = {kk: v for kk, v in r.items() if not kk.startswith("_") and kk != "trace"}
            out.append({"_key": k, **r["_spec"], **body})
        return pd.DataFrame(out)

    def traces(self, keys=None):
        out = [{"_key": k, **r["_spec"], **t} for k, r in self.recs.items()
               if keys is None or k in keys for t in r.get("trace", [])]
        return pd.DataFrame(out)


def cached(store, spec, fn, label=None, cleanup=None):
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if cleanup is not None:          # only after the record is on disk
        cleanup()
    if label:
        print(f"  [{store.block}] {label}  done in {payload['_elapsed'] / 60:.1f} min")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    tmp, prev = p.with_suffix(".tmp"), p.with_suffix(".prev")
    torch.save(obj, tmp)
    if p.exists():
        os.replace(p, prev)
    os.replace(tmp, p)


def ckpt_load(name):
    if not RESUME:
        return None
    p = ckpt_path(name)
    for cand in (p, p.with_suffix(".prev")):
        if cand.exists():
            try:
                return torch.load(cand, map_location=DEVICE, weights_only=False)
            except Exception as e:
                print(f"  checkpoint {cand.name} unreadable ({type(e).__name__}); trying the previous one")
    return None


def ckpt_delete(name):
    p = ckpt_path(name)
    for cand in (p, p.with_suffix(".prev"), p.with_suffix(".tmp")):
        if cand.exists():
            cand.unlink()


print(f"results {RESDIR}\ncheckpoints {CKPTDIR}")


## Data

TinyStories is tokenised once with the GPT-2 tokenizer and remapped to the 7,999 most frequent
tokens plus UNK, which keeps the embedding and softmax small enough for a T4. Recall sequences
use reserved key and value tokens outside the language vocabulary:

`SEP k1 v1 k2 v2 … kL vL SEP  ? q1  ? q2 …`

The value is supervised at the position of each queried key; chance is 1/256.


In [ ]:
def _local_stories(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        cur = []
        for line in f:
            while "<|endoftext|>" in line:
                head, _, line = line.partition("<|endoftext|>")
                cur.append(head)
                s = "".join(cur).strip()
                if s:
                    yield s
                cur = []
            cur.append(line)


def _stories(n_train, n_val=5000):
    if LOCAL_TEXT is not None:
        it = _local_stories(LOCAL_TEXT)
        val = list(itertools.islice(it, n_val))
        return itertools.islice(it, n_train), val
    from datasets import load_dataset
    tr = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    va = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
    return ((ex["text"] for ex in itertools.islice(tr, n_train)),
            [ex["text"] for ex in itertools.islice(va, n_val)])


def _encode(texts, tok):
    enc = tok(texts, add_special_tokens=False)["input_ids"]
    eos = tok.eos_token_id
    return np.concatenate([np.asarray(e + [eos], dtype=np.uint16) for e in enc])


def prepare_data():
    stem = f"tinystories_{N_STORIES}_v{CFG.lm_vocab}"
    ftr, fva, fmeta = (DATADIR / f"{stem}_{s}" for s in ("train.npy", "val.npy", "meta.json"))
    if ftr.exists() and fva.exists() and fmeta.exists():
        return np.load(ftr, mmap_mode="r"), np.load(fva, mmap_mode="r"), json.loads(fmeta.read_text())
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("gpt2")
    tr_iter, va_texts = _stories(N_STORIES)
    t0, chunks, buf, n = time.time(), [], [], 0
    for s in tr_iter:
        buf.append(s); n += 1
        if len(buf) == 2000:
            chunks.append(_encode(buf, tok)); buf = []
            if n % 100_000 == 0:
                print(f"  tokenised {n:,} stories  {(time.time() - t0) / 60:.1f} min")
    if buf:
        chunks.append(_encode(buf, tok))
    raw_tr, raw_va = np.concatenate(chunks), _encode(va_texts, tok)
    counts = np.bincount(raw_tr.astype(np.int64), minlength=len(tok))
    keep = np.argsort(-counts, kind="stable")[: CFG.lm_vocab - 1]
    lut = np.zeros(len(tok), dtype=np.uint16)
    lut[keep] = np.arange(1, CFG.lm_vocab, dtype=np.uint16)
    np.save(ftr, lut[raw_tr]); np.save(fva, lut[raw_va])
    meta = {"stories": n, "train_tokens": int(len(raw_tr)), "val_tokens": int(len(raw_va)),
            "coverage": float(counts[keep].sum() / counts.sum())}
    fmeta.write_text(json.dumps(meta))
    return np.load(ftr, mmap_mode="r"), np.load(fva, mmap_mode="r"), meta


TRAIN, VAL, META = prepare_data()
print(f"train {META['train_tokens']:,} tokens from {META['stories']:,} stories, "
      f"val {META['val_tokens']:,}; top-{CFG.lm_vocab - 1} vocabulary covers "
      f"{100 * META['coverage']:.2f}% of tokens")


def lm_batch(B, data, rng):
    T = CFG.seq_len
    ix = rng.integers(0, len(data) - T - 1, size=B)
    arr = np.stack([np.asarray(data[i:i + T + 1], dtype=np.int64) for i in ix])
    t = torch.from_numpy(arr).to(DEVICE, non_blocking=True)
    return t[:, :-1].contiguous(), t[:, 1:].contiguous()


def recall_batch(B, load, gen):
    # Built on the generator's device (CUDA for training, CPU for the fixed evaluation sets,
    # which keeps evaluation data identical across sessions and GPU types).
    dev, T, nq = gen.device, CFG.seq_len, min(CFG.n_query, load)
    keys = torch.rand(B, CFG.n_keys, generator=gen, device=dev).argsort(-1)[:, :load]
    vals = torch.randint(0, CFG.n_vals, (B, load), generator=gen, device=dev)
    qidx = torch.rand(B, load, generator=gen, device=dev).argsort(-1)[:, :nq]
    x = torch.full((B, T), PAD, dtype=torch.long, device=dev)
    y = torch.full((B, T), IGNORE, dtype=torch.long, device=dev)
    x[:, 0] = SEP
    x[:, 1:1 + 2 * load] = torch.stack([KEY0 + keys, VAL0 + vals], -1).reshape(B, 2 * load)
    p = 1 + 2 * load
    x[:, p] = SEP
    qpos = p + 1 + 2 * torch.arange(nq, device=dev)        # positions of the "?" tokens
    x[:, qpos] = QRY
    x[:, qpos + 1] = KEY0 + keys.gather(1, qidx)
    y[:, qpos + 1] = VAL0 + vals.gather(1, qidx)            # predict the value at the key
    return x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)


def train_batch(rng, gen):
    xl, yl = lm_batch(CFG.batch_lm, TRAIN, rng)
    per = CFG.batch_rec // CFG.rec_groups
    xs, ys = [], []
    for L in rng.choice(LOADS, size=CFG.rec_groups):
        x, y = recall_batch(per, int(L), gen); xs.append(x); ys.append(y)
    return torch.cat([xl] + xs), yl, torch.cat(ys)


def _show(tok):
    t = int(tok)
    if t == PAD: return "_"
    if t == SEP: return "|"
    if t == QRY: return "?"
    if t >= VAL0: return f"v{t - VAL0}"
    if t >= KEY0: return f"k{t - KEY0}"
    return f"w{t}"


_g = torch.Generator(device="cpu"); _g.manual_seed(0)
_x, _y = recall_batch(1, 4, _g)
_n = 2 + 2 * 4 + 2 * 4
print("recall example (load 4):", " ".join(_show(t) for t in _x[0, :_n]))
print("targets at             :", [(_i, _show(int(_y[0, _i]))) for _i in range(_n) if _y[0, _i] != IGNORE])


def make_eval_sets():
    g = torch.Generator(device="cpu"); g.manual_seed(1234)
    rec = {L: [recall_batch(64, L, g) for _ in range(N_EVAL_BATCHES)] for L in LOADS}
    rng = np.random.default_rng(1234)
    lm = [lm_batch(32, VAL, rng) for _ in range(N_EVAL_LM)]
    return rec, lm


EVAL_REC, EVAL_LM = make_eval_sets()
print(f"evaluation: {N_EVAL_BATCHES * 64} recall sequences per load, {N_EVAL_LM * 32} LM sequences")


## Model and route interventions

Interventions are passed as a dict, `iv`:

- `{"attn": ("cache", clean_writes, perm)}` replaces every layer's attention write with the write
  that another sequence (`perm`) produced in its own clean forward pass. This is route-level
  resample ablation.
- `{"attn": ("mask", keep)}` zeroes the attention write for rows where `keep` is False. This is the
  stochastic availability used in training.
- `{"no_attn": True}` skips attention entirely (deletion, no KV cache).
- The same keys exist for `"rec"`.


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)


def rope_cache(T, dh, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2).float() / dh))
    f = torch.outer(torch.arange(T).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):                     # x: B, heads, T, dh
    x1, x2 = x[..., ::2], x[..., 1::2]
    cos, sin = cos[None, None].to(x.dtype), sin[None, None].to(x.dtype)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.dh = n_heads, d // n_heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, D))


def log_g(x):                                    # log of g(x) = x + 0.5 (x >= 0), sigmoid(x) (x < 0)
    return torch.where(x >= 0, torch.log(F.relu(x) + 0.5), -F.softplus(-x))


class Recurrence(nn.Module):
    # h_t = (1 - z_t) * h_{t-1} + z_t * g(h~_t), computed with a parallel log-space scan
    # (minGRU). No convolution, so recurrence does not hand attention a previous-token
    # binding for free; the output gate follows Griffin/Hawk. The state is int(d * rnn_expand)
    # per layer, and the branch has almost exactly the parameter count of the attention branch.
    def __init__(self, d, expand=1.0):
        super().__init__()
        r = int(d * expand)
        self.inp = nn.Linear(d, 3 * r)
        self.norm = RMSNorm(r)
        self.o = nn.Linear(r, d, bias=False)

    def scan(self, x):
        z_, h_, g_ = self.inp(x).float().chunk(3, -1)
        with torch.autocast(device_type=DEVICE.type, enabled=False):
            log_a = -F.softplus(z_)                  # log(1 - z)
            log_b = -F.softplus(-z_) + log_g(h_)     # log z + log g(h~)
            a_star = torch.cumsum(log_a, 1)
            h = torch.exp(a_star + torch.logcumsumexp(log_b - a_star, 1))
        return h, g_

    def forward(self, x):
        h, g_ = self.scan(x)
        return self.o(self.norm(h) * F.gelu(g_))


def route_out(v, spec, layer):
    if v is None or spec is None:
        return v
    kind = spec[0]
    if kind == "cache":
        return spec[1][layer][spec[2]]
    if kind == "mask":
        return v * spec[1].to(v.dtype)[:, None, None]
    if kind == "mean":
        return v.mean(0, keepdim=True).expand_as(v)
    raise ValueError(kind)


class Block(nn.Module):
    def __init__(self, cfg, attention=True, recurrence=True):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d), RMSNorm(cfg.d)
        self.attn = Attention(cfg.d, cfg.n_heads) if attention else None
        self.rec = Recurrence(cfg.d, cfg.rnn_expand) if recurrence else None
        self.mlp = nn.Sequential(nn.Linear(cfg.d, cfg.mlp_mult * cfg.d, bias=False), nn.GELU(),
                                 nn.Linear(cfg.mlp_mult * cfg.d, cfg.d, bias=False))

    def forward(self, x, cos, sin, iv, layer, collect=None, cache=None):
        h = self.n1(x)
        a = self.attn(h, cos, sin) if (self.attn is not None and not iv.get("no_attn")) else None
        r = self.rec(h) if (self.rec is not None and not iv.get("no_rec")) else None
        if cache is not None:
            if a is not None: cache["attn"].append(a)
            if r is not None: cache["rec"].append(r)
        a, r = route_out(a, iv.get("attn"), layer), route_out(r, iv.get("rec"), layer)
        if collect is not None:
            if a is not None: collect["attn"].append(a.float().norm(dim=-1))
            if r is not None: collect["rec"].append(r.float().norm(dim=-1))
        if a is not None: x = x + a
        if r is not None: x = x + r
        return x + self.mlp(self.n2(x))


class HybridLM(nn.Module):
    def __init__(self, cfg, attention=True, recurrence=True):
        super().__init__()
        self.cfg, self.attn_deleted = cfg, False
        self.emb = nn.Embedding(VOCAB, cfg.d)
        self.blocks = nn.ModuleList([Block(cfg, attention, recurrence) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d)
        self.head = nn.Linear(cfg.d, VOCAB, bias=False)
        self.head.weight = self.emb.weight
        cos, sin = rope_cache(cfg.seq_len, cfg.d // cfg.n_heads)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        for mod in self.modules():
            if isinstance(mod, (nn.Linear, nn.Embedding)):
                nn.init.normal_(mod.weight, std=0.02)
            if isinstance(mod, nn.Linear) and mod.bias is not None:
                nn.init.zeros_(mod.bias)

    @property
    def has_attn(self):
        return self.blocks[0].attn is not None and not self.attn_deleted

    @property
    def has_rec(self):
        return self.blocks[0].rec is not None

    def forward(self, x, iv=None, collect=None, cache=None):
        iv = dict(iv or {})
        if self.attn_deleted:
            iv["no_attn"] = True
        T = x.shape[1]
        h = self.emb(x)
        for i, b in enumerate(self.blocks):
            h = b(h, self.cos[:T], self.sin[:T], iv, i, collect, cache)
        return self.head(self.norm(h))


def count_params(m):
    return sum(p.numel() for p in m.parameters())


for _name, _kw in [("hybrid", {}), ("recurrent-only", {"attention": False})]:
    _m = HybridLM(CFG, **_kw)
    print(f"{_name:<15} {count_params(_m) / 1e6:6.2f}M params, "
          f"{(count_params(_m) - _m.emb.weight.numel()) / 1e6:6.2f}M non-embedding")
del _m


In [ ]:
def make_opt(m):
    return torch.optim.AdamW(m.parameters(), lr=CFG.lr, betas=(0.9, 0.95),
                             weight_decay=CFG.weight_decay)


def make_scaler(state=None):
    sc = torch.amp.GradScaler("cuda", enabled=IS_CUDA and AMP_DTYPE == torch.float16)
    if state:                    # a disabled scaler saves {}, which an enabled one cannot load
        sc.load_state_dict(state)
    return sc


def task_loss(logits, yl, yr):
    Bl, lg = yl.shape[0], logits.float()
    lm = F.cross_entropy(lg[:Bl].reshape(-1, VOCAB), yl.reshape(-1))
    rc = F.cross_entropy(lg[Bl:].reshape(-1, VOCAB), yr.reshape(-1), ignore_index=IGNORE)
    return lm + CFG.recall_w * rc, lm, rc


@torch.no_grad()
def evaluate(model):
    was_training = model.training
    model.eval()
    out = {}
    for L in LOADS:
        hits = dict.fromkeys(["clean", "attn_res", "rec_res", "no_attn", "no_rec"], 0.0)
        n, an, rn = 0, 0.0, 0.0
        for x, y in EVAL_REC[L]:
            sel = y != IGNORE
            tgt = y[sel]
            n += int(sel.sum())

            def hit(iv, cache=None, coll=None):
                with autocast():
                    lg = model(x, iv, coll, cache)
                return float((lg.argmax(-1)[sel] == tgt).sum())

            cache, coll = {"attn": [], "rec": []}, {"attn": [], "rec": []}
            hits["clean"] += hit({}, cache, coll)
            nonpad = (x != PAD).float()
            an += sum(float((t * nonpad).sum()) for t in coll["attn"])
            rn += sum(float((t * nonpad).sum()) for t in coll["rec"])
            perm = torch.roll(torch.arange(x.shape[0], device=DEVICE), 1)   # every row gets another's
            if model.has_attn:
                hits["attn_res"] += hit({"attn": ("cache", cache["attn"], perm)})
                hits["no_attn"] += hit({"no_attn": True})
            if model.has_rec:
                hits["rec_res"] += hit({"rec": ("cache", cache["rec"], perm)})
                hits["no_rec"] += hit({"no_rec": True})
            del cache
        acc = hits["clean"] / n
        den = max(acc - CHANCE, 1e-6)
        dep = lambda k: float(np.clip((acc - hits[k] / n) / den, 0.0, 1.0))
        out[f"acc_L{L}"] = acc
        out[f"Dattn_L{L}"] = dep("attn_res") if model.has_attn else 0.0
        out[f"Drec_L{L}"] = dep("rec_res") if model.has_rec else 0.0
        out[f"acc_no_attn_L{L}"] = hits["no_attn"] / n if model.has_attn else acc
        out[f"acc_no_rec_L{L}"] = hits["no_rec"] / n if model.has_rec else acc
        out[f"attn_share_L{L}"] = an / (an + rn + 1e-9)      # contribution proxy, not evidence
    tot = {"lm": 0.0, "lm_no_attn": 0.0, "lm_no_rec": 0.0}
    for x, y in EVAL_LM:
        def ce(iv):
            with autocast():
                lg = model(x, iv)
            return float(F.cross_entropy(lg.float().reshape(-1, VOCAB), y.reshape(-1)))
        base = ce({})
        tot["lm"] += base
        tot["lm_no_attn"] += ce({"no_attn": True}) if model.has_attn else base
        tot["lm_no_rec"] += ce({"no_rec": True}) if model.has_rec else base
    out.update({k: v / len(EVAL_LM) for k, v in tot.items()})
    if was_training:
        model.train()
    return out


class SimulatedDisconnect(Exception):
    pass


def train_run(key, model, steps, seed, *, opt_state=None, scaler_state=None, warmup=False,
              lam=0.0, p_absent=0.0, log_every=0, log_extra=(), step_offset=0, stop_after=None,
              final_name=None):
    # Resumable training. A mid-run checkpoint (model, optimiser, scaler, step, trace, timing) is
    # written every CKPT_MINUTES; rerunning the same call resumes from it. The data stream after
    # a resume is re-seeded from (seed, key, step): statistically equivalent, not bit-identical.
    # With `final_name`, the finished state is saved BEFORE the mid-run checkpoint is deleted, and a
    # finished state found on disk is returned without retraining, so no window loses a run.
    if final_name is not None:
        fin = ckpt_load(final_name)
        if fin is not None and "trace" in fin:
            model.load_state_dict(fin["model"])
            opt = make_opt(model); opt.load_state_dict(fin["opt"])
            ckpt_delete(f"run_{key}")        # a leftover from a disconnect right after saving
            print("    finished run found on disk; not retraining")
            return model, opt, make_scaler(fin["scaler"]), fin["trace"], fin.get("stats", {})
    opt = make_opt(model)
    if opt_state is not None:
        opt.load_state_dict(opt_state)
    scaler = make_scaler(scaler_state)
    start, trace, done_time, done_steps = 1, [], 0.0, 0
    ck = ckpt_load(f"run_{key}")
    if ck is not None and ck.get("steps") == steps:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        scaler = make_scaler(ck["scaler"])
        start, trace = ck["step"] + 1, ck["trace"]
        done_time, done_steps = ck.get("train_time", 0.0), ck.get("train_steps", 0)
        print(f"    resumed at step {ck['step']}/{steps}")
    log_extra = set(log_extra)
    if start == 1 and (log_every or log_extra):
        trace.append({"step": step_offset, **evaluate(model)})
    rng = np.random.default_rng([seed, int(key[:8], 16), start])
    gen = torch.Generator(device=DEVICE); gen.manual_seed(int(rng.integers(2 ** 62)))
    model.train()
    sync(); t0, overhead, last_ck = time.time(), 0.0, time.time()
    for step in range(start, steps + 1):
        lr = CFG.lr * (min(1.0, step / CFG.warmup) if warmup else 1.0)
        for g in opt.param_groups:
            g["lr"] = lr
        x, yl, yr = train_batch(rng, gen)
        iv = {}
        if p_absent > 0:
            iv["attn"] = ("mask", torch.rand(x.shape[0], generator=gen, device=DEVICE) >= p_absent)
        coll = {"attn": [], "rec": []} if lam > 0 else None
        with autocast():
            logits = model(x, iv, coll)
        loss, lm, rc = task_loss(logits, yl, yr)
        if lam > 0:
            loss = loss + lam * torch.stack([t.mean() for t in coll["attn"]]).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt); scaler.update()

        if step == start + 19:
            sync(); dt = (time.time() - t0 - overhead) / 20
            print(f"    {dt * 1000:.0f} ms/step, about {dt * (steps - step) / 60:.1f} min left in this run")
        if (log_every and step % log_every == 0) or step in log_extra or step == steps:
            sync(); t = time.time()
            trace.append({"step": step_offset + step, "train_lm": float(lm), "train_recall": float(rc),
                          **evaluate(model)})
            sync(); overhead += time.time() - t
        if step < steps and (time.time() - last_ck > 60 * CKPT_MINUTES or step == stop_after):
            sync(); t = time.time()
            ckpt_save(f"run_{key}", {"model": model.state_dict(), "opt": opt.state_dict(),
                                     "scaler": scaler.state_dict(), "step": step, "steps": steps,
                                     "trace": trace, "train_time": done_time + (t - t0 - overhead),
                                     "train_steps": done_steps + (step - start + 1)})
            overhead += time.time() - t
            last_ck = time.time()
            if step == stop_after:
                raise SimulatedDisconnect(step)
    sync()
    total_time = done_time + (time.time() - t0 - overhead)
    total_steps = done_steps + (steps - start + 1)
    stats = {"ms_per_step": 1000.0 * total_time / max(total_steps, 1)}
    if final_name is not None:
        ckpt_save(final_name, {"model": model.state_dict(), "opt": opt.state_dict(),
                               "scaler": scaler.state_dict(), "trace": trace, "stats": stats})
    ckpt_delete(f"run_{key}")
    return model, opt, scaler, trace, stats


def mask_dependence(df):
    # D divides by accuracy headroom above chance; near chance it is noise (0/0-like), so tables,
    # summaries and figures show it as undefined there. Raw values stay in the JSONL records.
    if df is None or not len(df):
        return df
    df = df.copy()
    for L in LOADS:
        a = f"acc_L{L}"
        if a not in df:
            continue
        low = (df[a] - CHANCE) < D_FLOOR
        for k in ("Drec", "Dattn"):
            if f"{k}_L{L}" in df:
                df.loc[low, f"{k}_L{L}"] = np.nan
    return df


def free_cuda():
    if IS_CUDA:
        torch.cuda.empty_cache()


def load_table(rec, keys=("acc", "Drec", "Dattn", "acc_no_attn")):
    rows = {k: [rec.get(f"{k}_L{L}", np.nan) for L in LOADS] for k in keys}
    return pd.DataFrame(rows, index=[f"L{L}" for L in LOADS]).T.round(3).to_string()


In [ ]:
# Self-test: the parts every number depends on, including resume after a disconnect.
set_seed(0)
# 1. the parallel scan equals the sequential recurrence it claims to compute
_rec = Recurrence(16).to(DEVICE)
_xx = torch.randn(3, 20, 16, device=DEVICE)
with torch.no_grad():
    z_, h_, g_ = _rec.inp(_xx).float().chunk(3, -1)
    z, gh = torch.sigmoid(z_), torch.where(h_ >= 0, h_ + 0.5, torch.sigmoid(h_))
    h, hs = torch.zeros(3, 16, device=DEVICE), []
    for t in range(20):
        h = (1 - z[:, t]) * h + z[:, t] * gh[:, t]; hs.append(h)
    seq = _rec.o(_rec.norm(torch.stack(hs, 1)) * F.gelu(g_))
    assert torch.allclose(seq, _rec(_xx), atol=1e-4), "parallel scan != sequential recurrence"

_m = HybridLM(CFG).to(DEVICE).eval()
_g0 = torch.Generator(device="cpu"); _g0.manual_seed(0)
_x, _y = recall_batch(8, 4, _g0)
_id = torch.arange(8, device=DEVICE)
with torch.no_grad():
    _cache = {"attn": [], "rec": []}
    _clean = _m(_x, {}, None, _cache)
    assert len(_cache["attn"]) == CFG.n_layers and len(_cache["rec"]) == CFG.n_layers
    # 2. resampling with the identity permutation reproduces the clean pass exactly
    _same = _m(_x, {"attn": ("cache", _cache["attn"], _id), "rec": ("cache", _cache["rec"], _id)})
    assert torch.allclose(_clean, _same, atol=1e-5), "identity resample changed the output"
    # 3. a derangement and deletion both change the output
    assert not torch.allclose(_clean, _m(_x, {"attn": ("cache", _cache["attn"], torch.roll(_id, 1))}))
    assert not torch.allclose(_clean, _m(_x, {"no_attn": True}))
    # 4. causality: changing the last token cannot change earlier logits
    _x2 = _x.clone(); _x2[:, -1] = SEP
    assert torch.allclose(_clean[:, :-1], _m(_x2)[:, :-1], atol=1e-5), "information leaks backwards"
    # 5. an all-False availability mask equals deletion
    _k0 = torch.zeros(8, dtype=torch.bool, device=DEVICE)
    assert torch.allclose(_m(_x, {"attn": ("mask", _k0)}), _m(_x, {"no_attn": True}), atol=1e-5)
del _m, _rec, _cache

# 6. resume: stop mid-run with a checkpoint, restart with a differently initialised model, finish
_key = hashlib.md5(f"selftest-{TAG}".encode()).hexdigest()[:16]
ckpt_delete(f"run_{_key}")
set_seed(0)
try:
    train_run(_key, HybridLM(CFG).to(DEVICE), 30, 0, warmup=True, log_every=10, stop_after=15)
    raise AssertionError("simulated disconnect did not happen")
except SimulatedDisconnect:
    pass
assert ckpt_load(f"run_{_key}") is not None, "no checkpoint after the simulated disconnect"
set_seed(1)
_m, _, _, _tr, _st = train_run(_key, HybridLM(CFG).to(DEVICE), 30, 0, warmup=True, log_every=10)
assert [t["step"] for t in _tr] == [0, 10, 20, 30], [t["step"] for t in _tr]
assert ckpt_load(f"run_{_key}") is None, "mid-run checkpoint not cleaned up"
del _m
free_cuda()
print(f"self-test passed: scan, resample identity, derangement, deletion, causality, mask, resume "
      f"({_st['ms_per_step']:.0f} ms/step on this device)")


## Stage A: recurrent-only viability, then the free hybrid

Both run for every seed in `SEEDS`, whether or not interventions are allowed. The recurrent-only
model trains for `PRE + POST` steps, the same total budget a steered hybrid receives. Its per-load
accuracy therefore gives both the handover prediction and the scaffold comparison.

At the end of this stage, `VIABILITY_SUMMARY.txt` is printed and saved. **In the full preset, send
it back before setting `RUN_INTERVENTIONS=True`.** A usable split needs at least one solved load
where recurrence alone is viable and at least one where it is not.


In [ ]:
CFG_SPEC = {"cfg": asdict(CFG), "loads": LOADS}
CFG_HASH = keyof(CFG_SPEC)


def viability_spec(sd):
    return {"block": "viability", "seed": sd, "steps": PRE + POST, **CFG_SPEC}


def free_spec(sd):
    return {"block": "free", "seed": sd, "steps": PRE, **CFG_SPEC}


def free_state(sd):
    # The switch checkpoint: model, optimiser, scaler. Retrained only if it has been deleted.
    spec = free_spec(sd)
    name = f"free_{keyof(spec)}"
    ck = ckpt_load(name)
    if ck is None:
        set_seed(sd)
        m = HybridLM(CFG).to(DEVICE)
        train_run(keyof(spec), m, PRE, sd, warmup=True, log_every=LOG, final_name=name)
        del m
        free_cuda()
        ck = ckpt_load(name)
    return ck


def model_from(ck):
    m = HybridLM(CFG).to(DEVICE)
    m.load_state_dict(ck["model"])
    return m


st = Store("viability")
for sd in SEEDS:
    _vname = f"viab_{keyof(viability_spec(sd))}"

    def _go(sd=sd, name=_vname):
        set_seed(sd)
        m = HybridLM(CFG, attention=False).to(DEVICE)
        m, _, _, tr, stats = train_run(keyof(viability_spec(sd)), m, PRE + POST, sd,
                                       warmup=True, log_every=2 * LOG, final_name=name)
        out = {**evaluate(m), "params": count_params(m), "trace": tr, **stats}
        del m; free_cuda()
        return out

    r = cached(st, viability_spec(sd), _go, label=f"recurrent-only seed {sd}",
               cleanup=lambda name=_vname: ckpt_delete(name))
    print(f"recurrent-only seed {sd}: lm {r['lm']:.3f}\n{load_table(r, ('acc',))}\n")

st = Store("free")
for sd in SEEDS:
    def _go(sd=sd):
        ck = free_state(sd)
        m = model_from(ck)
        out = {**evaluate(m), "params": count_params(m),
               "nonemb_params": count_params(m) - m.emb.weight.numel(),
               "trace": ck["trace"], **ck.get("stats", {})}
        del m; free_cuda()
        return out

    r = cached(st, free_spec(sd), _go, label=f"free hybrid seed {sd}")
    print(f"free hybrid seed {sd} at step {PRE}: lm {r['lm']:.3f}, without attention "
          f"{r['lm_no_attn']:.3f}\n{load_table(r)}\n")


def current_keys(block_spec_fn, seeds=range(20)):
    return {keyof(block_spec_fn(sd)) for sd in seeds}


def viability_summary():
    V = Store("viability").frame(current_keys(viability_spec))
    FR = mask_dependence(Store("free").frame(current_keys(free_spec)))
    lines = [f"=== VIABILITY SUMMARY | preset {PRESET} | size {SIZE} | cfg {CFG_HASH} | {GPU_NAME} ===",
             f"rules: predicted handover if recurrent-only acc >= {VIABLE}; load excluded if free acc < {SOLVED}"]
    if not len(V) or not len(FR):
        lines.append("incomplete: need both recurrent-only and free-hybrid records")
        return {"text": "\n".join(lines), "verdict": "INCOMPLETE"}
    seeds = sorted(set(V.seed) & set(FR.seed))
    V, FR = V[V.seed.isin(seeds)].set_index("seed"), FR[FR.seed.isin(seeds)].set_index("seed")
    cols = [f"L{L}" for L in LOADS]
    tab = {}
    for sd in seeds:
        tab[f"rec-only s{sd}"] = [V.loc[sd, f"acc_L{L}"] for L in LOADS]
    for sd in seeds:
        tab[f"free acc s{sd}"] = [FR.loc[sd, f"acc_L{L}"] for L in LOADS]
    vm = {L: float(V[f"acc_L{L}"].mean()) for L in LOADS}
    fm = {L: float(FR[f"acc_L{L}"].mean()) for L in LOADS}
    tab["rec-only mean"] = [vm[L] for L in LOADS]
    tab["free acc mean"] = [fm[L] for L in LOADS]
    tab["free D_attn mean"] = [float(FR[f"Dattn_L{L}"].mean()) for L in LOADS]
    tab["free D_rec mean"] = [float(FR[f"Drec_L{L}"].mean()) for L in LOADS]
    tab["free noAttn acc"] = [float(FR[f"acc_no_attn_L{L}"].mean()) for L in LOADS]
    lines.append(pd.DataFrame(tab, index=cols).T.round(3).to_string())
    solved = [L for L in LOADS if fm[L] >= SOLVED]
    viable = [L for L in solved if vm[L] >= VIABLE]
    nonviable = [L for L in solved if vm[L] < VIABLE]
    near = [L for L in solved if abs(vm[L] - VIABLE) < NEAR]
    verdict = "OK" if (viable and nonviable) else "DEGENERATE"
    lines += [f"seeds {seeds} | lm: rec-only {V.lm.mean():.3f}, free {FR.lm.mean():.3f}, "
              f"free without attention {FR.lm_no_attn.mean():.3f}",
              f"ms/step: rec-only {V.get('ms_per_step', pd.Series([np.nan])).mean():.0f}, "
              f"free {FR.get('ms_per_step', pd.Series([np.nan])).mean():.0f}",
              f"solved loads {solved} | predicted handover {viable} | predicted no handover {nonviable}",
              f"borderline (within {NEAR} of {VIABLE}): {near or 'none'}",
              f"SPLIT VERDICT: {verdict}"]
    if verdict != "OK":
        lines.append("  no usable boundary: if recurrence is viable everywhere, add harder loads or lower "
                     "Cfg.rnn_expand; if nowhere, add easier loads or raise it. Changing either "
                     "starts fresh runs.")
    text = "\n".join(lines)
    (TABDIR / "VIABILITY_SUMMARY.txt").write_text(text)
    return {"text": text, "verdict": verdict, "seeds": seeds, "vm": vm, "fm": fm,
            "solved": solved, "viable": viable, "nonviable": nonviable, "near": near}


VS = viability_summary()
print(VS["text"])
if PRESET == "full" and not RUN_INTERVENTIONS:
    print("\nStage A finished. Send VIABILITY_SUMMARY.txt back; set RUN_INTERVENTIONS=True only "
          "once the split is confirmed. The remaining cells will skip the interventions.")


## Predictions, recorded before any intervention

`predictions.json` is written once: when interventions are first allowed and no file exists. It
records the seeds and configuration it was based on.
- Seeds added later are evaluated against this prediction; it is never recomputed.
- If the file exists but the configuration hash differs, the notebook stops.
- In the full preset, a degenerate split blocks the predictions (and so the arms) unless
  `ALLOW_DEGENERATE_SPLIT=True`.


In [ ]:
PRED_PATH = RESDIR / "predictions.json"
PRED = None

if PRED_PATH.exists():
    PRED = json.loads(PRED_PATH.read_text())
    if PRED.get("cfg_hash") != CFG_HASH:
        raise RuntimeError(
            f"predictions.json was recorded for config {PRED.get('cfg_hash')}, but the current config "
            f"is {CFG_HASH}. Restore the original configuration, or delete {PRED_PATH} deliberately "
            f"to start a new pre-registered experiment.")
    print(f"using predictions recorded {PRED['created']} from seeds {PRED['seeds_used']} "
          f"(never overwritten)")
elif INTERVENTIONS_ALLOWED:
    if VS["verdict"] == "INCOMPLETE":
        INTERVENTIONS_ALLOWED = False
        print("stage A incomplete; predictions not written")
    elif VS["verdict"] != "OK" and PRESET == "full" and not ALLOW_DEGENERATE_SPLIT:
        INTERVENTIONS_ALLOWED = False
        print("split is DEGENERATE; predictions not written and arms skipped "
              "(set ALLOW_DEGENERATE_SPLIT=True to override)")
    else:
        PRED = {"created": time.strftime("%Y-%m-%d %H:%M:%S %Z"), "unix_time": time.time(),
                "cfg_hash": CFG_HASH, "preset": PRESET, "size": SIZE, "seeds_used": VS["seeds"],
                "rule": f"handover predicted if recurrent-only acc >= {VIABLE}; "
                        f"loads with free acc < {SOLVED} excluded",
                "recurrent_only_acc": {str(L): VS["vm"][L] for L in LOADS},
                "free_acc": {str(L): VS["fm"][L] for L in LOADS},
                "predicted_handover": {str(L): bool(VS["vm"][L] >= VIABLE) for L in LOADS},
                "excluded_unsolved": [L for L in LOADS if VS["fm"][L] < SOLVED],
                "borderline": VS["near"], "split_verdict": VS["verdict"]}
        PRED_PATH.write_text(json.dumps(PRED, indent=1))
        print(f"predictions written {PRED['created']} from seeds {PRED['seeds_used']}")
else:
    print("interventions not allowed yet; predictions.json will be written when RUN_INTERVENTIONS=True")

if PRED is not None:
    for L in LOADS:
        tag = ("excluded (free hybrid unsolved)" if L in PRED["excluded_unsolved"]
               else "predict HANDOVER" if PRED["predicted_handover"][str(L)] else "predict no handover")
        print(f"  load {L:>2}: recurrent-only {PRED['recurrent_only_acc'][str(L)]:.3f}, "
              f"free {PRED['free_acc'][str(L)]:.3f} -> {tag}")


## Stage B: arms from the switch checkpoint, then withdrawal

Every arm loads the free model **and its optimiser state** at step `PRE`, then trains `POST`
steps at constant learning rate on the same data mixture:

| arm | what changes at the switch |
|---|---|
| `never` | nothing |
| `late_availability` | each step, a random `P_ABSENT` of rows get no attention write; no penalty term |
| `late_price` | `λ·mean‖attention write‖`, with λ set so the penalty is `PRICE_FRAC` of the task loss at the switch |
| `delete_finetune` | attention removed permanently, then fine-tuned |
| `withdrawn` | continues `late_availability` for `WITHDRAW` steps with no pressure |

Evaluations run at the switch, at steps `EARLY_POST` after it, and every `LOG_POST` steps. The
post-switch minimum accuracy uses only evaluations after the switch.


In [ ]:
ARMS = ["never", "late_availability", "late_price", "delete_finetune"]


def arm_spec(sd, arm):
    return {"block": "arm", "arm": arm, "seed": sd, "pre": PRE, "post": POST,
            "p_absent": P_ABSENT, "price_frac": PRICE_FRAC, **CFG_SPEC}


def withdrawn_spec(sd):
    return {"block": "withdrawn", "seed": sd, "from": keyof(arm_spec(sd, "late_availability")),
            "steps": WITHDRAW, **CFG_SPEC}


@torch.no_grad()
def calibrate_price(m, sd):
    rng = np.random.default_rng([sd, 99])
    gen = torch.Generator(device=DEVICE); gen.manual_seed(sd + 99)
    m.eval(); ce, pen = [], []
    for _ in range(4):
        x, yl, yr = train_batch(rng, gen)
        coll = {"attn": [], "rec": []}
        with autocast():
            lg = m(x, {}, coll)
        ce.append(float(task_loss(lg, yl, yr)[0]))
        pen.append(float(torch.stack([t.mean() for t in coll["attn"]]).mean()))
    m.train()
    return PRICE_FRAC * float(np.mean(ce)) / max(float(np.mean(pen)), 1e-6)


def run_arm(sd, arm):
    spec = arm_spec(sd, arm)
    ck = free_state(sd)
    m = model_from(ck)
    m.attn_deleted = arm == "delete_finetune"
    lam = calibrate_price(m, sd) if arm == "late_price" else 0.0
    m, opt, sc, tr, stats = train_run(
        keyof(spec), m, POST, sd + 1000, opt_state=ck["opt"], scaler_state=ck["scaler"], lam=lam,
        p_absent=P_ABSENT if arm == "late_availability" else 0.0,
        log_every=LOG_POST, log_extra=EARLY_POST, step_offset=PRE, final_name=f"arm_{keyof(spec)}")
    out = {**evaluate(m), "lam": lam, "trace": tr, **stats}
    del m, ck; free_cuda()
    return out


def run_withdrawn(sd):
    name = f"arm_{keyof(arm_spec(sd, 'late_availability'))}"
    if ckpt_load(name) is None:
        print("    late_availability checkpoint missing; regenerating it")
        run_arm(sd, "late_availability")
    ck = ckpt_load(name)
    m = model_from(ck)
    m, _, _, tr, stats = train_run(
        keyof(withdrawn_spec(sd)), m, WITHDRAW, sd + 2000, opt_state=ck["opt"],
        scaler_state=ck["scaler"], log_every=LOG_POST, log_extra=EARLY_POST,
        step_offset=PRE + POST, final_name=f"wd_{keyof(withdrawn_spec(sd))}")
    out = {**evaluate(m), "trace": tr, **stats}
    del m, ck; free_cuda()
    return out


if INTERVENTIONS_ALLOWED and PRED is not None:
    st_a, st_w = Store("arm"), Store("withdrawn")
    for sd in SEEDS:
        for arm in ARMS:
            _final = f"arm_{keyof(arm_spec(sd, arm))}"
            # late_availability's final state is kept: withdrawn continues from it
            _clean = None if arm == "late_availability" else (lambda n=_final: ckpt_delete(n))
            r = cached(st_a, arm_spec(sd, arm), lambda sd=sd, arm=arm: run_arm(sd, arm),
                       label=f"{arm:<18} seed {sd}", cleanup=_clean)
            print(f"{arm} seed {sd}: lm {r['lm']:.3f}, without attention {r['lm_no_attn']:.3f}, "
                  f"lambda {r['lam']:.4g}\n{load_table(r)}\n")
        r = cached(st_w, withdrawn_spec(sd), lambda sd=sd: run_withdrawn(sd), label=f"withdrawn seed {sd}",
                   cleanup=lambda n=f"wd_{keyof(withdrawn_spec(sd))}": ckpt_delete(n))
        print(f"withdrawn seed {sd}: lm {r['lm']:.3f}\n{load_table(r)}\n")
else:
    print("interventions skipped (stage A only, or predictions not written)")


In [ ]:
# Efficiency payoff. Memory numbers are analytic and exact for this architecture; throughput at
# the training length understates attention's quadratic cost, so quote the memory figures.
kv = 2 * CFG.n_layers * CFG.d * 2                          # K and V per token, fp16 bytes
state = CFG.n_layers * int(CFG.d * CFG.rnn_expand) * 2     # recurrent state, fp16 bytes, constant
_m = HybridLM(CFG).to(DEVICE).eval()
_xb = EVAL_LM[0][0]


@torch.no_grad()
def tokens_per_s(iv, reps=10):
    for _ in range(3):
        with autocast():
            _m(_xb, iv)
    sync(); t0 = time.time()
    for _ in range(reps):
        with autocast():
            _m(_xb, iv)
    sync()
    return _xb.numel() * reps / (time.time() - t0)


EFF = {"kv_cache_bytes_per_token": kv, "kv_cache_MB_at_4096_tokens": kv * 4096 / 2 ** 20,
       "kv_cache_MB_at_32768_tokens": kv * 32768 / 2 ** 20, "recurrent_state_bytes_total": state,
       "tokens_per_s_hybrid": tokens_per_s({}), "tokens_per_s_without_attention": tokens_per_s({"no_attn": True})}
(TABDIR / "efficiency.json").write_text(json.dumps(EFF, indent=1))
for k, v in EFF.items():
    print(f"  {k:<32} {v:,.1f}")
del _m; free_cuda()


## Report: tables, figures, compact summary

Definitions used in every table:
- **Handover** for a seed at a load: D_rec > D_attn at the end of the arm, with accuracy at least
  `RETAIN` × that seed's free-model accuracy.
- **Post-switch minimum:** the lowest recall accuracy among evaluations strictly after the switch,
  per seed. Tables report the mean of these minima and the worst seed.
- **No-attention accuracy:** recall with every attention branch removed, the most interpretable
  single number in a fused model.

Only records matching the current configuration are used.


In [ ]:
ARM_KEYS = {keyof(arm_spec(sd, a)) for sd in range(20) for a in ARMS}
WD_KEYS = current_keys(withdrawn_spec)
V_ALL = Store("viability").frame(current_keys(viability_spec))
FREE_ALL = mask_dependence(Store("free").frame(current_keys(free_spec)))
ARM = mask_dependence(Store("arm").frame(ARM_KEYS))
WD = mask_dependence(Store("withdrawn").frame(WD_KEYS))
HAVE_ARMS = len(ARM) > 0 and PRED is not None
KEYS = ("acc", "Drec", "Dattn", "acc_no_attn", "attn_share")
ORDER = ["never", "late_availability", "late_price", "delete_finetune", "withdrawn"]


def f3(v):
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"


if not HAVE_ARMS:
    print("no intervention results for this configuration yet; tables and figures need stage B")
else:
    SEEDS_ARM = sorted(int(s) for s in ARM.seed.unique())
    V = V_ALL[V_ALL.seed.isin(SEEDS_ARM)]
    FREE = FREE_ALL[FREE_ALL.seed.isin(SEEDS_ARM)]
    fr_seed = FREE.set_index("seed")

    rows = []
    for L in LOADS:
        r = {"load": L, "predicted": bool(PRED["predicted_handover"][str(L)]),
             "solved": L not in PRED["excluded_unsolved"],
             "rec_only_acc": float(V[f"acc_L{L}"].mean()) if len(V) else np.nan}
        for k in KEYS:
            r[f"free_{k}"] = float(FREE[f"{k}_L{L}"].mean())
        for arm in ARMS:
            g = ARM[ARM.arm == arm].set_index("seed")
            if not len(g):
                continue
            for k in KEYS:
                r[f"{arm}_{k}"] = float(g[f"{k}_L{L}"].mean())
                r[f"{arm}_{k}_sd"] = float(g[f"{k}_L{L}"].std()) if len(g) > 1 else 0.0
            ok = ((g[f"Drec_L{L}"] > g[f"Dattn_L{L}"])
                  & (g[f"acc_L{L}"] >= RETAIN * fr_seed[f"acc_L{L}"].reindex(g.index)))
            r[f"{arm}_handover"], r[f"{arm}_n"] = int(ok.sum()), int(len(g))
        if len(WD):
            for k in KEYS:
                r[f"withdrawn_{k}"] = float(WD[f"{k}_L{L}"].mean())
        rows.append(r)
    SUMMARY = pd.DataFrame(rows)
    SUMMARY.to_csv(TABDIR / "scale_summary.csv", index=False)
    S = SUMMARY.set_index("load")

    # post-switch minima (strictly after the switch; the step-PRE evaluation is excluded)
    TA = mask_dependence(Store("arm").traces(ARM_KEYS))
    TW = mask_dependence(Store("withdrawn").traces(WD_KEYS))
    parts = [TA[TA.step > PRE]]
    if len(TW):
        parts.append(TW[TW.step > PRE + POST].assign(arm="withdrawn"))
    TPOST = pd.concat(parts, ignore_index=True)
    agg = {**{f"acc_L{L}": "min" for L in LOADS}, "lm": "max"}
    per_seed = TPOST.groupby(["arm", "seed"]).agg(agg).reset_index()
    MIN_MEAN = per_seed.groupby("arm").mean(numeric_only=True).drop(columns="seed").reindex(
        [a for a in ORDER if a in set(per_seed.arm)])
    MIN_WORST = per_seed.groupby("arm").agg({**{f"acc_L{L}": "min" for L in LOADS}, "lm": "max"}).reindex(MIN_MEAN.index)
    MIN_MEAN.to_csv(TABDIR / "scale_post_switch_min_mean.csv")
    MIN_WORST.to_csv(TABDIR / "scale_post_switch_min_worst.csv")
    at_del = TA[(TA.arm == "delete_finetune") & (TA.step == PRE)]
    ACC_AT_DELETION = {L: float(at_del[f"acc_L{L}"].mean()) for L in LOADS} if len(at_del) else {}

    lm_rows = [{"condition": f"free hybrid (step {PRE})", "lm": FREE.lm.mean(),
                "lm_no_attn": FREE.lm_no_attn.mean(), "max_post_switch": np.nan, "n": len(FREE)}]
    for arm in ARMS:
        g = ARM[ARM.arm == arm]
        if len(g):
            lm_rows.append({"condition": arm, "lm": g.lm.mean(), "lm_no_attn": g.lm_no_attn.mean(),
                            "max_post_switch": MIN_MEAN.loc[arm, "lm"] if arm in MIN_MEAN.index else np.nan,
                            "n": len(g)})
    if len(WD):
        lm_rows.append({"condition": "withdrawn", "lm": WD.lm.mean(), "lm_no_attn": WD.lm_no_attn.mean(),
                        "max_post_switch": MIN_MEAN.loc["withdrawn", "lm"] if "withdrawn" in MIN_MEAN.index else np.nan,
                        "n": len(WD)})
    if len(V):
        lm_rows.append({"condition": f"recurrent-only ({PRE + POST} steps)", "lm": V.lm.mean(),
                        "lm_no_attn": V.lm.mean(), "max_post_switch": np.nan, "n": len(V)})
    LMT = pd.DataFrame(lm_rows)
    LMT.to_csv(TABDIR / "scale_lm.csv", index=False)

    CHK = None
    if "late_availability_n" in S:
        CHK = SUMMARY[SUMMARY.solved][["load", "predicted", "late_availability_handover", "late_availability_n"]].copy()
        CHK["observed"] = CHK.late_availability_handover > CHK.late_availability_n / 2
        CHK["match"] = CHK.predicted == CHK.observed
        CHK.to_csv(TABDIR / "scale_prediction_check.csv", index=False)

    # ---- LaTeX ------------------------------------------------------------------------------
    tot, nonemb = FREE.params.iloc[0] / 1e6, FREE.nonemb_params.iloc[0] / 1e6
    has = lambda arm: f"{arm}_n" in S
    tex = [r"% generated by hybrid_lm_scale_colab.ipynb -- requires booktabs",
           r"\begin{table}[t]", r"\centering", r"\small",
           rf"\caption{{\textbf{{Post-convergence steering in a parallel hybrid language model}} "
           rf"({tot:.1f}M parameters, {nonemb:.1f}M non-embedding; {len(SEEDS_ARM)} seeds). "
           rf"Rec.-only: recurrent-only model trained {PRE + POST} steps; accuracy $\geq {VIABLE}$ "
           rf"defines the handover prediction, recorded before any intervention. Late availability "
           rf"removes the attention write on {int(100 * P_ABSENT)}\% of rows for {POST} steps after the "
           rf"step-{PRE} switch. Handover: seeds with $D_h > D_a$ and accuracy $\geq {RETAIN}\times$ the "
           rf"free model.}}",
           r"\label{tab:scale}", r"\begin{tabular}{rcccccccc}", r"\toprule",
           r" & & & free & \multicolumn{3}{c}{late availability} & \multicolumn{2}{c}{no-attention acc.} \\",
           r"\cmidrule(lr){5-7}\cmidrule(lr){8-9}",
           r"load & rec.-only & pred. & acc. & acc. & $D_h$ / $D_a$ & handover & never & late \\",
           r"\midrule"]
    for L in LOADS:
        r = S.loc[L]
        pred = "excl." if not r["solved"] else (r"\checkmark" if r["predicted"] else "--")
        hv = (f"{int(r['late_availability_handover'])}/{int(r['late_availability_n'])}"
              if has("late_availability") else "--")
        cells = [str(L), f3(r["rec_only_acc"]), pred, f3(r["free_acc"]), f3(r.get("late_availability_acc")),
                 f"{f3(r.get('late_availability_Drec'))} / {f3(r.get('late_availability_Dattn'))}", hv,
                 f3(r.get("never_acc_no_attn")), f3(r.get("late_availability_acc_no_attn"))]
        tex.append(" & ".join(cells) + r" \\")
    tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]

    tex += [r"\begin{table}[t]", r"\centering", r"\small",
            rf"\caption{{Post-switch minimum recall accuracy: mean over seeds of each seed's minimum across "
            rf"all evaluations after the switch (worst seed in parentheses), and the largest "
            rf"post-switch validation LM loss.}}",
            r"\label{tab:scale-min}", r"\begin{tabular}{l" + "c" * (len(LOADS) + 1) + "}", r"\toprule",
            "arm & " + " & ".join(f"load {L}" for L in LOADS) + r" & max LM loss \\", r"\midrule"]
    for arm in MIN_MEAN.index:
        cells = [arm.replace("_", r"\_")]
        cells += [f"{MIN_MEAN.loc[arm, f'acc_L{L}']:.3f} ({MIN_WORST.loc[arm, f'acc_L{L}']:.3f})" for L in LOADS]
        cells.append(f"{MIN_MEAN.loc[arm, 'lm']:.3f}")
        tex.append(" & ".join(cells) + r" \\")
    tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]

    tex += [r"\begin{table}[t]", r"\centering", r"\small",
            r"\caption{Validation language-model loss (nats/token) at the end of each condition, "
            r"with attention deleted, and the largest post-switch value.}",
            r"\label{tab:scale-lm}", r"\begin{tabular}{lcccc}", r"\toprule",
            r"condition & loss & loss w/o attention & max post-switch & seeds \\", r"\midrule"]
    for _, r in LMT.iterrows():
        tex.append(" & ".join([str(r["condition"]).replace("_", r"\_"), f3(float(r["lm"])),
                               f3(float(r["lm_no_attn"])), f3(float(r["max_post_switch"])),
                               str(int(r["n"]))]) + r" \\")
    tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TABDIR / "scale_tables.tex").write_text("\n".join(tex))

    print("per load, end of each arm (means over seeds):")
    show = ["solved", "predicted", "rec_only_acc", "free_acc", "free_Drec", "free_Dattn", "free_acc_no_attn"]
    for arm in ARMS:
        show += [c for c in (f"{arm}_acc", f"{arm}_Drec", f"{arm}_Dattn", f"{arm}_acc_no_attn",
                             f"{arm}_attn_share", f"{arm}_handover") if c in S]
    show += [c for c in ("withdrawn_acc", "withdrawn_Drec", "withdrawn_Dattn") if c in S]
    print(S[show].T.round(3).to_string())
    print("\npost-switch minimum recall accuracy (mean of per-seed minima):")
    print(MIN_MEAN.round(3).to_string())
    print("\nvalidation LM loss:")
    print(LMT.round(4).to_string(index=False))
    if CHK is not None:
        print(f"\nprediction check (late availability, majority of seeds): "
              f"{int(CHK.match.sum())}/{len(CHK)} solved loads match")
        print(CHK.to_string(index=False))
    print(f"\nwrote {TABDIR / 'scale_tables.tex'} and CSVs")


In [ ]:
if HAVE_ARMS:
    mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight",
                         "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
                         "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
                         "axes.spines.top": False, "axes.spines.right": False})
    COL = {"free": "#8d99ae", "never": "#1b6ca8", "late_availability": "#e07a1f",
           "late_price": "#3a9d5d", "delete_finetune": "#6a4c93", "withdrawn": "#d1495b",
           "recurrent-only": "#222222"}
    LABEL = {"free": "free (before switch)", "never": "never", "late_availability": "late availability",
             "late_price": "late price", "delete_finetune": "delete + fine-tune", "withdrawn": "withdrawn"}
    TF = mask_dependence(Store("free").traces(current_keys(free_spec, SEEDS_ARM)))

    def band(ax, df, col, color, ls="-", label=None):
        if df is None or not len(df) or col not in df:
            return
        g = df.groupby("step")[col].agg(["mean", "std"]).reset_index()
        sd = g["std"].fillna(0)
        ax.plot(g.step, g["mean"], color=color, ls=ls, lw=1.2, label=label)
        ax.fill_between(g.step, g["mean"] - sd, g["mean"] + sd, color=color, alpha=0.15, lw=0)

    def arm_df(arm):
        if arm == "withdrawn":
            return TW if len(TW) else None
        return TA[TA.arm == arm]

    solved = [L for L in LOADS if L not in PRED["excluded_unsolved"]]
    vi = [L for L in solved if PRED["predicted_handover"][str(L)]]
    nv = [L for L in solved if not PRED["predicted_handover"][str(L)]]
    SEL = sorted(set(([min(vi), max(vi)] if vi else []) + ([min(nv)] if nv else []))) or LOADS[:2]
    tag = lambda L: "predicted handover" if L in vi else ("excluded" if L not in solved else "predicted no handover")

    def savefig(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(FIGDIR / f"{name}.{ext}")
        plt.show()

    # 1. causal dependence trajectories with seed bands
    fig, axes = plt.subplots(2, len(SEL), figsize=(2.3 * len(SEL), 3.8), sharex=True, sharey=True, squeeze=False)
    for j, L in enumerate(SEL):
        for i, (key, name) in enumerate([("Dattn", "attention dependence $D_a$"),
                                         ("Drec", "recurrent dependence $D_h$")]):
            ax, col = axes[i, j], f"{key}_L{L}"
            band(ax, TF, col, COL["free"], label=LABEL["free"])
            for arm in ["never", "late_availability", "late_price", "withdrawn"]:
                band(ax, arm_df(arm), col, COL[arm], ls="--" if arm == "withdrawn" else "-", label=LABEL[arm])
            for s in (PRE, PRE + POST):
                ax.axvline(s, color="0.6", lw=0.6, ls=":")
            ax.set_ylim(-0.03, 1.03)
            if i == 0:
                ax.set_title(f"load {L} ({tag(L)})")
            if j == 0:
                ax.set_ylabel(name)
            if i == 1:
                ax.set_xlabel("training step")
    h, lab = axes[0, 0].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h, lab, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 0.0))
    savefig(fig, "scale_dependence_trajectories")

    # 2. behaviour through the transition: recall accuracy and LM loss
    x0 = PRE - 2 * LOG_POST
    fig, axes = plt.subplots(1, len(SEL) + 1, figsize=(2.3 * (len(SEL) + 1), 2.3), squeeze=False)
    for j, L in enumerate(SEL + ["lm"]):
        ax = axes[0, j]
        col = "lm" if L == "lm" else f"acc_L{L}"
        band(ax, TF[TF.step >= x0] if len(TF) else None, col, COL["free"], label=LABEL["free"])
        for arm in ["never", "late_availability", "late_price", "delete_finetune", "withdrawn"]:
            band(ax, arm_df(arm), col, COL[arm], ls="--" if arm == "withdrawn" else "-", label=LABEL[arm])
        for s in (PRE, PRE + POST):
            ax.axvline(s, color="0.6", lw=0.6, ls=":")
        ax.set_xlim(left=x0)
        ax.set_title("validation LM loss" if L == "lm" else f"recall accuracy, load {L}")
        ax.set_xlabel("training step")
    h, lab = axes[0, 0].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h, lab, loc="upper center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 0.0))
    savefig(fig, "scale_behaviour_trajectories")

    # 3. accuracy by load, clean and with attention deleted
    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.4), sharey=True)
    conds = [("free", f"free (step {PRE})")] + [(a, LABEL[a]) for a in ARMS if f"{a}_acc" in S]
    if "withdrawn_acc" in S:
        conds.append(("withdrawn", LABEL["withdrawn"]))
    for ax, suffix, title in [(axes[0], "acc", "recall accuracy"),
                              (axes[1], "acc_no_attn", "accuracy with attention deleted")]:
        for c, lab in conds:
            ax.plot(LOADS, S[f"{c}_{suffix}"], "-o", ms=3, color=COL[c], label=lab)
        if len(V):
            ax.plot(LOADS, S["rec_only_acc"], "-s", ms=3, color=COL["recurrent-only"],
                    label=f"recurrent-only ({PRE + POST} steps)")
        ax.set_xscale("log", base=2); ax.set_xticks(LOADS); ax.set_xticklabels(LOADS)
        ax.axhline(CHANCE, color="0.7", lw=0.6, ls=":")
        ax.set_title(title); ax.set_xlabel("key-value pairs")
    axes[0].set_ylabel("accuracy")
    h, lab = axes[0].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h, lab, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
    savefig(fig, "scale_accuracy_by_load")
    print(f"figures written to {FIGDIR}")


In [ ]:
# Compact results summary to paste back, plus an auto-filled draft paragraph (numbers only).
def _cell(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "--"
    return f"{v:.3f}" if isinstance(v, float) else str(v)


def _fmt_row(label, vals, w=7):
    return f"{label:<26}" + "".join(_cell(v).rjust(w) for v in vals)


if HAVE_ARMS:
    load_hdr = f"{'':<26}" + "".join(f"{'L' + str(L):>7}" for L in LOADS)
    out = [f"=== HYBRID LM SCALE CHECK: RESULTS SUMMARY | {time.strftime('%Y-%m-%d %H:%M')} ===",
           f"preset {PRESET} | size {SIZE} | cfg {CFG_HASH} | {GPU_NAME} | amp {str(AMP_DTYPE).split('.')[-1]} "
           f"| torch {torch.__version__}",
           f"params {FREE.params.iloc[0] / 1e6:.2f}M ({FREE.nonemb_params.iloc[0] / 1e6:.2f}M non-emb) | "
           f"pre {PRE}, post {POST}, withdraw {WITHDRAW} | seeds with arms {SEEDS_ARM}",
           f"ms/step: rec-only {V.get('ms_per_step', pd.Series([np.nan])).mean():.0f}, "
           f"free {FREE.get('ms_per_step', pd.Series([np.nan])).mean():.0f}, "
           f"arms {ARM.get('ms_per_step', pd.Series([np.nan])).mean():.0f}",
           f"predictions: {PRED['created']} from seeds {PRED['seeds_used']} | verdict {PRED['split_verdict']} | "
           f"excluded {PRED['excluded_unsolved']} | borderline {PRED['borderline']}",
           f"D shown as -- where accuracy is within {D_FLOOR} of chance ({CHANCE:.4f})",
           "", "-- stage A (means over seeds with arms) --", load_hdr,
           _fmt_row("rec-only acc", [S.loc[L, "rec_only_acc"] for L in LOADS]),
           _fmt_row("predicted handover", ["yes" if S.loc[L, "predicted"] else "no" for L in LOADS]),
           _fmt_row("free acc", [S.loc[L, "free_acc"] for L in LOADS]),
           _fmt_row("free D_rec", [S.loc[L, "free_Drec"] for L in LOADS]),
           _fmt_row("free D_attn", [S.loc[L, "free_Dattn"] for L in LOADS]),
           _fmt_row("free acc no-attn", [S.loc[L, "free_acc_no_attn"] for L in LOADS]),
           _fmt_row("free attn share", [S.loc[L, "free_attn_share"] for L in LOADS])]
    for arm in ARMS + ["withdrawn"]:
        if f"{arm}_acc" not in S:
            continue
        out += ["", f"-- {arm} (end) --", load_hdr]
        for k, lab in [("acc", "acc"), ("Drec", "D_rec"), ("Dattn", "D_attn"),
                       ("acc_no_attn", "acc no-attn"), ("attn_share", "attn share")]:
            out.append(_fmt_row(lab, [S.loc[L, f"{arm}_{k}"] for L in LOADS]))
        if f"{arm}_Drec_sd" in S and len(SEEDS_ARM) > 1:
            out.append(_fmt_row("  sd D_rec", [S.loc[L, f"{arm}_Drec_sd"] for L in LOADS]))
            out.append(_fmt_row("  sd D_attn", [S.loc[L, f"{arm}_Dattn_sd"] for L in LOADS]))
        if f"{arm}_handover" in S:
            out.append(_fmt_row("handover seeds", [f"{int(S.loc[L, f'{arm}_handover'])}/{int(S.loc[L, f'{arm}_n'])}" for L in LOADS]))
        if arm in MIN_MEAN.index:
            out.append(_fmt_row("post-switch min acc", [MIN_MEAN.loc[arm, f"acc_L{L}"] for L in LOADS]))
            out.append(_fmt_row("  worst seed", [MIN_WORST.loc[arm, f"acc_L{L}"] for L in LOADS]))
        if arm == "delete_finetune" and ACC_AT_DELETION:
            out.append(_fmt_row("acc right after deletion", [ACC_AT_DELETION[L] for L in LOADS]))
    lam = ARM[ARM.arm == "late_price"].lam.mean() if (ARM.arm == "late_price").any() else np.nan
    out += ["", f"-- LM validation loss (late_price lambda {lam:.4g}) --", LMT.round(4).to_string(index=False)]
    if CHK is not None:
        out += ["", f"-- prediction check: {int(CHK.match.sum())}/{len(CHK)} solved loads match (majority of seeds) --",
                CHK.to_string(index=False)]
    out += ["", f"-- efficiency -- KV cache {EFF['kv_cache_MB_at_4096_tokens']:.1f} MB at 4k tokens, "
                f"{EFF['kv_cache_MB_at_32768_tokens']:.1f} MB at 32k; recurrent state "
                f"{EFF['recurrent_state_bytes_total'] / 1024:.1f} KB; tokens/s {EFF['tokens_per_s_hybrid']:,.0f} "
                f"(hybrid) vs {EFF['tokens_per_s_without_attention']:,.0f} (no attention) at length {CFG.seq_len}"]
    RESULTS_SUMMARY = "\n".join(out)
    (TABDIR / "RESULTS_SUMMARY.txt").write_text(RESULTS_SUMMARY)
    print(RESULTS_SUMMARY)

    # ---- draft paragraph: numbers only; interpretation left as TODO ------------------------
    def lst(xs):
        xs = [str(x) for x in xs]
        return "none" if not xs else (xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1])

    def counts(arm, loads):
        return lst([f"{int(S.loc[L, arm + '_handover'])}/{int(S.loc[L, arm + '_n'])} at load {L}" for L in loads])

    def vals(col, loads):
        return lst([f"{S.loc[L, col]:.2f} (load {L})" for L in loads])

    solved = [L for L in LOADS if bool(S.loc[L, "solved"])]
    vi = [L for L in solved if bool(S.loc[L, "predicted"])]
    nv = [L for L in solved if not bool(S.loc[L, "predicted"])]
    lmd = LMT.set_index("condition")
    free_lm = lmd.loc[f"free hybrid (step {PRE})", "lm"]
    d = [r"% AUTO-GENERATED DRAFT (numbers only). Check every value against tables/ before use.",
         r"\subsection{Scale check: a parallel hybrid language model}",
         f"To test whether the phenomenon and its capacity-limited boundary replicate beyond the "
         f"identifiable instrument, we train a parallel recurrent--attention language model "
         f"({FREE.params.iloc[0] / 1e6:.1f}M parameters, {FREE.nonemb_params.iloc[0] / 1e6:.1f}M non-embedding, "
         f"{CFG.n_layers} layers, $d={CFG.d}$) from scratch on TinyStories jointly with in-context associative "
         f"recall at {lst(LOADS)} key--value pairs. Each block sends one normalised input to a gated linear "
         f"recurrence and to causal attention and adds both writes to the residual stream. Route dependence "
         f"resamples every layer's attention (or recurrent) write from another sequence's clean forward pass "
         f"and is normalised as in Eq.~(2); results use {len(SEEDS_ARM)} seeds.",
         f"Before any intervention, a recurrent-only model trained for {PRE + POST} steps reached accuracy "
         f"$\\geq {VIABLE}$ at loads {lst(vi)} and not at {lst(nv)}; these were recorded as predicted handover "
         f"and non-handover loads. At the step-{PRE} switch the free hybrid had validation loss {free_lm:.3f} and "
         f"attention dependence {vals('free_Dattn', solved)}."]
    if "late_availability_n" in S:
        la_min = lst([f"{MIN_MEAN.loc['late_availability', f'acc_L{L}']:.2f} (load {L})" for L in solved]) \
            if "late_availability" in MIN_MEAN.index else "n/a"
        d.append(f"Late stochastic attention unavailability ($p={P_ABSENT}$, no penalty term) for {POST} steps "
                 f"produced handover in {counts('late_availability', vi)} seeds at predicted-handover loads and "
                 f"{counts('late_availability', nv)} at predicted non-handover loads. Recall accuracy never fell "
                 f"below {la_min} (mean of per-seed post-switch minima), and validation loss moved from "
                 f"{free_lm:.3f} to {lmd.loc['late_availability', 'lm']:.3f} "
                 f"(never-priced arm: {lmd.loc['never', 'lm']:.3f}). % TODO: interpretation")
        d.append(f"With attention deleted after steering, recall accuracy was "
                 f"{vals('late_availability_acc_no_attn', solved)}, versus {vals('never_acc_no_attn', solved)} for "
                 f"the never-priced arm; language-model loss without attention was "
                 f"{lmd.loc['late_availability', 'lm_no_attn']:.3f} versus {lmd.loc['never', 'lm_no_attn']:.3f}. "
                 f"% TODO: KV-cache sentence from tables/efficiency.json")
    if "late_price_n" in S:
        d.append(f"Pricing the attention write instead changed its contribution share from "
                 f"{vals('free_attn_share', solved)} to {vals('late_price_attn_share', solved)}, with attention "
                 f"dependence {vals('late_price_Dattn', solved)}. % TODO: relation to the Section 8 dissociation")
    if "delete_finetune_n" in S:
        d.append(f"Deleting attention at the switch and fine-tuning for the same {POST} steps reached "
                 f"{vals('delete_finetune_acc', solved)}. % TODO: compare with late availability")
    if len(WD):
        d.append(f"Removing the availability pressure for {WITHDRAW} steps gave attention dependence "
                 f"{vals('withdrawn_Dattn', solved)}. % TODO: reopening, partial, or none")
    d.append(r"% TODO: limitation -- trained from scratch; not a pretrained production hybrid.")
    (TABDIR / "scale_section_draft.tex").write_text("\n\n".join(d))

    for sub in (RESDIR, TABDIR, FIGDIR):
        shutil.make_archive(str(WORK / f"{sub.name}"), "zip", sub)
    print(f"\nsaved RESULTS_SUMMARY.txt, scale_tables.tex, scale_section_draft.tex; zipped results, "
          f"tables and figures into {WORK}")
else:
    print(VS["text"] if "VS" in globals() else "run stage A first")


## Using the outputs in the paper

**Files.**
- `tables_S_full/scale_tables.tex`: main table, post-switch minimum table, LM table (booktabs).
- `figures_S_full/`:
  - `scale_dependence_trajectories.pdf` (main-text candidate)
  - `scale_behaviour_trajectories.pdf` (appendix: accuracy and LM loss through the switch)
  - `scale_accuracy_by_load.pdf` (deletion payoff)
- `results_S_full/predictions.json`: quote its timestamp in the appendix as evidence that the
  predictions preceded every intervention record.
- `tables_S_full/RESULTS_SUMMARY.txt`: paste back for interpretation.

**Framing.** A conceptual replication in a larger fused hybrid trained jointly on natural text and
recall. Not the original experiment scaled up, and not a pretrained production model.

**Claims should follow the table.**
- Handover at predicted loads only: the phenomenon and its capacity-limited boundary replicate.
- Dependence moves without crossing: report the weaker claim.
- `late_price` shrinks attention share while D_attn stays high: the Section 8 dissociation at scale.
- `delete_finetune` matches steering: report it, and rest the argument on reversibility and
  load-dependent route choice.
